In [2]:
import banco
import pandas as pd


1. Os 5 órgãos com maior custo total?

In [ ]:
query1 = """
select nome_orgao_superior, 
    sum(valor_total) as valor_total
from silver_viagem
group by nome_orgao_superior
order by valor_total desc
limit 5
"""

conexao = banco.conectar()

orgao_maior_custo, colunas = banco.executar(conexao, query1)

df_orgao_maior_custo = pd.DataFrame(orgao_maior_custo, columns=colunas)

df_orgao_maior_custo

,nome_orgao_superior,valor_total
0,Ministério da Justiça e Segurança Pública,490813474.85
1,Ministério da Defesa,157059698.59
2,Ministério da Educação,112519456.96
3,Ministério do Meio Ambiente e Mudança do Clima,50548377.44
4,Ministério da Previdência Social,40921476.10


1. quais viagens urgentes custam mais por dia do que as não urgentes?
    - silver_viagem, agrupando por viagem_urgente
    - métricas: qtd de viagens, duração média em dias, valor total médio e custo médio diario de cada grupo
    - gráfico: barras verticais, comparando o custo médio diário dos dois grupos, com a quantidade de viagens indicadas no rótulo
    - análise: a urgência encarece o dia da viagem? o grupo urgente é revelante em volume ou é exceção dentro do total de gastos?

In [ ]:
# Verificando a média do custo médio diário das viagens urgentes e não urgentes 

query2 = """
SELECT COUNT(DISTINCT(id_viagem)) AS qtd_viagem,
    ROUND(AVG(duracao_dias),0) AS duracao_media_em_dias,
    ROUND(AVG(valor_total),2) AS valor_total_medio,
    ROUND(AVG(custo_medio_diario),2) AS media_custo_diario,
    viagem_urgente
FROM silver_viagem
GROUP BY viagem_urgente;
"""

conexao = banco.conectar()

media_urgente_nao_urgente, colunas = banco.executar(conexao, query2)

df_media_urgente_nao_urgente = pd.DataFrame(media_urgente_nao_urgente, columns=colunas)

df_media_urgente_nao_urgente

,qtd_viagem,duracao_media_em_dias,valor_total_medio,media_custo_diario,viagem_urgente
0,137911,7,2992.49,448.74,NÃO
1,203949,9,3864.02,604.49,SIM


In [ ]:
# verificando as viagens urgentes que possui custo médio diário maior que a média geral do custo diário das viagens não urgentes

query3 = """
SELECT id_viagem,
    duracao_dias,
    valor_total,
    custo_medio_diario,
    viagem_urgente
FROM silver_viagem
WHERE viagem_urgente = 'SIM'
HAVING custo_medio_diario > (
	SELECT AVG(custo_medio_diario)
    FROM silver_viagem
    WHERE viagem_urgente <> 'SIM')
ORDER BY custo_medio_diario DESC;
"""

conexao = banco.conectar()

viagens_urgentes, colunas = banco.executar(conexao,query3)

df_viagens_urgentes = pd.DataFrame(viagens_urgentes, columns=colunas)

df_viagens_urgentes

,id_viagem,duracao_dias,valor_total,custo_medio_diario,viagem_urgente
0,0000000000021054792,1,48953.46,48953.46,SIM
1,0000000000020643519,1,30842.51,30842.51,SIM
2,0000000000020961803,2,56302.53,28151.27,SIM
3,0000000000020925587,6,151696.96,25282.83,SIM
4,0000000000020968291,7,161725.38,23103.63,SIM
...,...,...,...,...,...
53856,0000000000021091981,13,5834.20,448.78,SIM
53857,0000000000020968327,4,1795.10,448.78,SIM
53858,0000000000020782403,20,8975.39,448.77,SIM
53859,0000000000020750910,6,2692.50,448.75,SIM


2. Como o custo médio diário varia conforme a duração da viagem?
    - silver_viagem, com faixas de duração dias criadas por CASE WHEN (1 dia, 2 a 3 dias, 4 a 7 dias, 8 a 15, acima de 15)
    - métricas: quantidade de viagens por faixa, custo médio diário e valor total médio
    - gráfico: barras por faixa de duração para custo médio diário. opcionalmente, uma linha sobreposta com a quantidade de viagens em eixo secundário
    - análise: há diluição do custo diário nas viagens longas, já que a passagem se distribui por mais dias? em que faixa está o dia de viagens mais caro?

In [ ]:
# verificando o comportamento do custo medio diario por categorias de duracao das viagens (1, 2 a 3, 4 a 7, 8 a 15, 15+)

query4 = """
SELECT 
	CASE 
		WHEN duracao_dias = 1 THEN '1 dia'
        WHEN duracao_dias BETWEEN 2 AND 3 THEN '2 a 3 dias'
        WHEN duracao_dias BETWEEN 4 AND 7 THEN '4 a 7 dias'
        WHEN duracao_dias BETWEEN 8 AND 15 THEN '8 a 15 dias'
        WHEN duracao_dias > 15 THEN 'acima de 15 dias'
	END AS categoria_duracao,
	COUNT(id_viagem) AS qtd_viagens,
    ROUND(AVG(custo_medio_diario),2) AS custo_medio_diario,
    ROUND(AVG(valor_total),2) AS media_valor_total
FROM silver_viagem
GROUP BY categoria_duracao
ORDER BY custo_medio_diario DESC
"""

conexao = banco.conectar()

custo_duracao_viagem, colunas = banco.executar(conexao, query4)

df_custo_duracao_viagem = pd.DataFrame(custo_duracao_viagem, columns=colunas)

df_custo_duracao_viagem

,categoria_duracao,qtd_viagens,custo_medio_diario,media_valor_total
0,2 a 3 dias,108236,586.41,1462.09
1,4 a 7 dias,121542,552.52,2825.74
2,8 a 15 dias,33874,550.58,5768.29
3,1 dia,53979,509.82,509.82
4,acima de 15 dias,24229,345.70,19652.05


<h1> Criando camada gold - tabela e view

3. Como o valor pago evoluiu mês a mês e qual tipo de pagamento sustenta essa evolução?
    - gold_resumo_pagamento_mensais (ou a VIEW), por ano, mês e tipo de pagamento
    - métricas: valor total pago por mês e por tipo de pagamento e a participação percentual de cada tipo no mês
    - gráfico: linhas com uma série por tipo de pagamento e os meses no eixo x. como alternativa, barras empilhadas para evidenciar a composição
    - análise: há sazonalidade no período? algum tipo de pagamento ganha ou perde participação ao longo dos meses?

In [12]:
# criando a view do resumo_pagamentos_mensais

drop_view = """
DROP VIEW IF EXISTS vw_resumo_pagamentos_mensais
"""

conexao = banco.conectar()

drop = banco.executar(conexao, drop_view)

In [ ]:
# removendo view se já existir

drop_view = """
DROP VIEW IF EXISTS vw_resumo_pagamentos_mensais
"""

conexao = banco.conectar()

drop = banco.executar(conexao, drop_view)

# criando a view do resumo_pagamentos_mensais e visualizando 10 linhas

create_view = """
CREATE VIEW vw_resumo_pagamentos_mensais AS
SELECT
	YEAR(sv.data_inicio) AS ano_referencia,
	MONTH(sv.data_inicio) AS mes_referencia,
	sp.nome_orgao_pagador,
	sp.tipo_pagamento,
	COUNT(DISTINCT(sv.id_viagem)) AS qtd_viagens,
	COUNT(DISTINCT(sp.id_pagamento)) AS qtd_pagamentos,
	ROUND(SUM(sp.valor),2) AS valor_total_pago,
	ROUND(AVG(sp.valor),2) AS valor_medio_pagamento
FROM silver_viagem as sv
JOIN silver_pagamento AS sp
ON sv.id_viagem = sp.id_viagem
GROUP BY YEAR(sv.data_inicio),
	MONTH(sv.data_inicio),
	sp.nome_orgao_pagador,
	sp.tipo_pagamento
"""

select_view = """
SELECT * FROM vw_resumo_pagamentos_mensais
LIMIT 10
"""

conexao = banco.conectar()

vw_resumo_pagamentos_mensais = banco.executar(conexao, create_view) # criar view

select_vw_resumo_pagamentos_mensais, colunas = banco.executar(conexao, select_view) # executar select

df_resumo_pagamentos_mensais = pd.DataFrame(select_vw_resumo_pagamentos_mensais, columns=colunas)

df_resumo_pagamentos_mensais


,ano_referencia,mes_referencia,nome_orgao_pagador,tipo_pagamento,qtd_viagens,qtd_pagamentos,valor_total_pago,valor_medio_pagamento
0,2025,1,Advocacia-Geral da União - Unidades com víncul...,DIÁRIAS,53,56,113748.68,2031.23
1,2025,1,Advocacia-Geral da União - Unidades com víncul...,PASSAGEM,5,7,56143.86,8020.55
2,2025,1,Advocacia-Geral da União - Unidades com víncul...,RESTITUIÇÃO,2,2,1200.55,600.28
3,2025,1,Advocacia-Geral da União - Unidades com víncul...,Serviço correlato: seguro,3,3,1756.39,585.46
4,2025,1,Agência Espacial Brasileira,DIÁRIAS,5,5,20408.60,4081.72
5,2025,1,Agência Espacial Brasileira,PASSAGEM,3,6,19983.52,3330.59
6,2025,1,Agência Espacial Brasileira,Serviço correlato: seguro,1,1,965.98,965.98
7,2025,1,Agência Nacional de Águas e Saneamento Básico,DIÁRIAS,7,7,9769.85,1395.69
8,2025,1,Agência Nacional de Águas e Saneamento Básico,PASSAGEM,6,8,16292.05,2036.51
9,2025,1,Agência Nacional de Aviação Civil,DIÁRIAS,231,249,412842.84,1658.00


4. Qual o perfil de gastos dos órgãos pagadores?
    - gold_resumo_pagamento_mensais somada ao longo dos meses e agrupada por nome_orgao_pagador, limitada aos 10 maiores por valor total pago.
    - métricas: valor total pago, quantidade de pagamentos, valor médio por pagamento e quantidade de viagens atendidas
    - gráfico: dispersão (bolhas), com quantidade de pagamentos no eixo x, o valor médio por pagamento no eixo y e o tamanho da bolha proporcional
ao valor total pago. Como alternativa, barras horizontais com valor total e ticket médio.
    - Análise: separe os órgãos de alto volume e ticket baixo dos de baixo volume e ticket alto, relacionando com a quantidade de viagens atendidas.

""
